# Metastatic Cancer Detection — v3: 4-model ensemble, clinical threshold, pseudo-labelling
**Deep Learning PBL · Saif Khattab (202312556) · Al-Zaytoonah University of Jordan**

v2 showed that bigger backbones raise the *internal* AUC (≈0.997) but not the Kaggle private AUC (≈0.968):
the remaining error is **domain shift between slides/labs**, not model capacity. v3 attacks that directly.

| Part | What | Why |
|---|---|---|
| A | **4-model rank ensemble** (v1 Baseline CNN + EfficientNetB0, v2 EfficientNetB2 + DenseNet121) | cheap, more diverse models |
| B | **Clinical threshold** chosen for ≥99% sensitivity on validation, reported on internal test | in screening a missed tumour costs far more than a false alarm; v1/v2's 95%-recall threshold (≈0.72) produced *more* missed tumours than 0.5 |
| C | **Pseudo-labelling** (semi-supervised): Kaggle test images on which the v2 models are very confident get hard labels and are added to training; v2 models are fine-tuned on train + pseudo with **stronger stain/colour augmentation** | lets the network adapt to the stain/scanner appearance of the unseen slides. Uses **no** ground-truth test labels (no PCam label leak) |

**Honesty note for the report:** pseudo-labelling is *transductive* (it uses the unlabelled test images). It is a standard,
legitimate semi-supervised technique, but it must be disclosed, and internal validation cannot measure its benefit
(validation slides overlap training slides) — only the Kaggle score can.

Inputs attached: competition data, output of *PBL Histopathologic Cancer Detection* (v1) and *PBL Cancer Detection v2*.
Run as **Save & Run All (Commit)**, GPU T4 x2.

## 1. Setup

In [ ]:
import os, glob, time, random, json, gc, traceback
RUN_START = time.time()
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from scipy.stats import rankdata
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import roc_auc_score, confusion_matrix, roc_curve

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED, IMG_SIZE, BATCH_SIZE = 42, 96, 128
# ---- pseudo-labelling ----
PL_POS, PL_NEG = 0.97, 0.03      # teacher confidence needed for a pseudo label
PL_LR          = 3e-5            # gentle fine-tuning from the v2 weights
PL_MAX_EPOCHS  = 6
PL_PATIENCE    = 2
MIN_LR         = 1e-7
# ---- stronger stain / colour augmentation than v2 (0.08 / 0.02) ----
HED_ALPHA, HED_BETA = 0.15, 0.03
SAT_RANGE      = (0.8, 1.2)
BRIGHT, CONTRAST = 0.08, (0.85, 1.15)
# ---- clinical operating point ----
TARGET_SENS    = 0.99
TOTAL_BUDGET_H, RESERVE_AFTER_H = 10.5, 0.5
STUDENTS = ['EfficientNetB2', 'DenseNet121']
OUT_DIR = '/kaggle/working'

keras.mixed_precision.set_global_policy('mixed_float16')
def set_seed(s):
    random.seed(s); np.random.seed(s); tf.random.set_seed(s)
set_seed(SEED)
def hours_left():
    return TOTAL_BUDGET_H - (time.time() - RUN_START) / 3600
print('TF', tf.__version__, '| GPUs:', tf.config.list_physical_devices('GPU'))

# locate attached inputs without walking the 220k competition images
def find_file(name):
    for root, dirs, files in os.walk('/kaggle/input'):
        dirs[:] = [d for d in dirs if d not in ('train', 'test')]
        if name in files:
            return os.path.join(root, name)
    raise FileNotFoundError(f'{name} not found - attach the v1 and v2 notebook outputs as inputs')
DATA_DIR = os.path.dirname(find_file('train_labels.csv'))
V1 = {k: find_file(k) for k in ['val_predictions.csv', 'test_predictions_all.csv', 'submission.csv', 'submission_baseline.csv']}
V2 = {k: find_file(k) for k in ['val_pred_efficientnetb2.csv', 'val_pred_densenet121.csv', 'test_pred_efficientnetb2.csv',
                                 'test_pred_densenet121.csv', 'submission_efficientnetb2.csv', 'submission_densenet121.csv']}
for k, v in {**V1, **V2}.items():
    print(f'{k:32s} <- {v}')

## 2. Helpers

In [ ]:
def rank_avg(preds):
    return np.mean([rankdata(p) / len(p) for p in preds], axis=0)

def evaluate(y, p, thr):
    pred = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    sens, spec = tp / (tp + fn), tn / (tn + fp)
    prec = tp / (tp + fp) if tp + fp else 0.0
    return {'threshold': round(float(thr), 4), 'AUC': roc_auc_score(y, p), 'Accuracy': (tp + tn) / len(y),
            'Sensitivity': sens, 'Specificity': spec, 'Precision': prec,
            'F1': 2 * prec * sens / (prec + sens) if prec + sens else 0.0,
            'FN (missed tumours)': int(fn), 'FP (false alarms)': int(fp)}

def threshold_for_sensitivity(y, p, target):
    # HIGHEST threshold whose sensitivity is still >= target (fewest false alarms at that safety level)
    fpr, tpr, thr = roc_curve(y, p)
    ok = np.where(tpr >= target)[0]
    return float(min(thr[ok[0]], 1.0))

val_df  = pd.read_csv(V2['val_pred_efficientnetb2.csv'])[['id', 'label']]
test_df = pd.read_csv(V2['test_pred_efficientnetb2.csv'])[['id', 'label']]
def load_val_test(kind):
    d = (val_df if kind == 'val' else test_df).copy()
    v1 = pd.read_csv(V1['val_predictions.csv'] if kind == 'val' else V1['test_predictions_all.csv'])
    cols = {'val': ('p_Baseline CNN', 'p_EfficientNetB0'), 'test': ('p_baseline', 'p_efficientnet')}[kind]
    v1 = v1[['id', cols[0], cols[1]]].rename(columns={cols[0]: 'Baseline CNN', cols[1]: 'EfficientNetB0'})
    d = d.merge(v1, on='id', how='left')
    for name, tag in [('EfficientNetB2', 'efficientnetb2'), ('DenseNet121', 'densenet121')]:
        v2 = pd.read_csv(V2[f'{kind}_pred_{tag}.csv'])[['id', 'p']].rename(columns={'p': name})
        d = d.merge(v2, on='id', how='left')
    assert d.notna().all().all(), f'{kind}: ids do not match between v1 and v2 (different split?)'
    return d
VAL, TEST = load_val_test('val'), load_val_test('test')
MODELS4 = ['Baseline CNN', 'EfficientNetB0', 'EfficientNetB2', 'DenseNet121']
print('val', VAL.shape, '| test', TEST.shape, '| split identical across v1/v2: OK')

## 3. Part A — ensembles of the existing models (validation used for selection, internal test for reporting)

In [ ]:
from itertools import combinations
rows = []
for k in range(1, 5):
    for combo in combinations(MODELS4, k):
        pv = rank_avg([VAL[m].values for m in combo]) if k > 1 else VAL[combo[0]].values
        pt = rank_avg([TEST[m].values for m in combo]) if k > 1 else TEST[combo[0]].values
        rows.append({'models': ' + '.join(combo), 'n': k,
                     'val AUC': roc_auc_score(VAL['label'], pv), 'test AUC': roc_auc_score(TEST['label'], pt)})
ens_table = pd.DataFrame(rows).sort_values('val AUC', ascending=False).reset_index(drop=True)
display(ens_table.round(5))
ens_table.to_csv(f'{OUT_DIR}/ensemble_combinations.csv', index=False)

# Kaggle test predictions of all 4 models, aligned on the same ids
KAG = pd.read_csv(V2['submission_efficientnetb2.csv']).rename(columns={'label': 'EfficientNetB2'})
for path, name in [(V2['submission_densenet121.csv'], 'DenseNet121'), (V1['submission.csv'], 'EfficientNetB0'),
                   (V1['submission_baseline.csv'], 'Baseline CNN')]:
    KAG = KAG.merge(pd.read_csv(path).rename(columns={'label': name}), on='id', how='left')
assert KAG.notna().all().all() and len(KAG) == 57458, KAG.shape
print('Spearman correlation between models on Kaggle test:')
display(KAG[MODELS4].corr(method='spearman').round(3))

best_combo = ens_table.loc[0, 'models'].split(' + ')
pd.DataFrame({'id': KAG['id'], 'label': rank_avg([KAG[m].values for m in MODELS4])}).to_csv(f'{OUT_DIR}/submission_ens4.csv', index=False)
pd.DataFrame({'id': KAG['id'], 'label': rank_avg([KAG[m].values for m in best_combo]) if len(best_combo) > 1 else KAG[best_combo[0]]}
             ).to_csv(f'{OUT_DIR}/submission_ens_bestval.csv', index=False)
print('Written submission_ens4.csv (all 4) and submission_ens_bestval.csv ->', best_combo)

## 4. Part B — clinical operating threshold
AUC ignores the threshold, but a deployed screening tool needs one. We use the **probability average** of the v2 models
(rank averages are not calibrated), choose on **validation** the highest threshold that still catches ≥99% of tumours (capped at 0.5),
and report it on the untouched internal test set next to the default 0.5.

In [ ]:
pv = VAL[['EfficientNetB2', 'DenseNet121']].mean(axis=1).values
pt = TEST[['EfficientNetB2', 'DenseNet121']].mean(axis=1).values
# capped at 0.5: a 'safety' threshold must never miss MORE tumours than the default
thr_clin = min(threshold_for_sensitivity(VAL['label'].values, pv, TARGET_SENS), 0.5)
clin = pd.DataFrame({
    'val @0.5': evaluate(VAL['label'].values, pv, 0.5),
    f'val @{thr_clin:.3f} (sens>={TARGET_SENS})': evaluate(VAL['label'].values, pv, thr_clin),
    'test @0.5': evaluate(TEST['label'].values, pt, 0.5),
    f'test @{thr_clin:.3f}': evaluate(TEST['label'].values, pt, thr_clin)})
display(clin.round(4))
clin.to_csv(f'{OUT_DIR}/clinical_threshold.csv')
# trade-off curve: missed tumours vs false alarms on internal test
ths = np.linspace(0.01, 0.99, 99)
fn = [((pt < t) & (TEST['label'].values == 1)).sum() for t in ths]
fp = [((pt >= t) & (TEST['label'].values == 0)).sum() for t in ths]
plt.figure(figsize=(7, 4)); plt.plot(ths, fn, label='missed tumours (FN)'); plt.plot(ths, fp, label='false alarms (FP)')
plt.axvline(thr_clin, ls='--', c='k', label=f'clinical thr {thr_clin:.3f}'); plt.axvline(0.5, ls=':', c='gray', label='0.5')
plt.yscale('log'); plt.xlabel('threshold'); plt.legend(); plt.title('Internal test: FN vs FP trade-off (v2 ensemble)'); plt.show()

## 5. Part C — pseudo-labelling: load images

In [ ]:
df = pd.read_csv(os.path.join(DATA_DIR, 'train_labels.csv')).reset_index(drop=True)
tr_df, temp_df = train_test_split(df, test_size=0.30, stratify=df['label'], random_state=SEED)
va_df, te_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df['label'], random_state=SEED)
assert (va_df['id'].values == VAL['id'].values).all() and (te_df['id'].values == TEST['id'].values).all(), 'split mismatch'

def load_images(ids, folder, workers=32):
    t0 = time.time()
    out = np.empty((len(ids), IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
    paths = [os.path.join(folder, f'{i}.tif') for i in ids]
    def work(k):
        with Image.open(paths[k]) as im:
            out[k] = np.asarray(im.convert('RGB'), dtype=np.uint8)
    with ThreadPoolExecutor(max_workers=workers) as ex:
        list(ex.map(work, range(len(paths))))
    print(f'  loaded {len(paths):,} images in {time.time() - t0:.0f}s')
    return out

kag_ids = KAG['id'].values
X_train = load_images(tr_df['id'].values, os.path.join(DATA_DIR, 'train'))
X_val   = load_images(va_df['id'].values,  os.path.join(DATA_DIR, 'train'))
X_test  = load_images(te_df['id'].values,  os.path.join(DATA_DIR, 'train'))
X_kag   = load_images(kag_ids,             os.path.join(DATA_DIR, 'test'))
y_train, y_val, y_test = (d['label'].values.astype('float32') for d in (tr_df, va_df, te_df))

# teacher = probability average of the two v2 models (best Kaggle models so far)
teacher = KAG[['EfficientNetB2', 'DenseNet121']].mean(axis=1).values
agree = (KAG['EfficientNetB2'].values >= 0.5) == (KAG['DenseNet121'].values >= 0.5)
sel = ((teacher >= PL_POS) | (teacher <= PL_NEG)) & agree
X_pl, y_pl = X_kag[sel], (teacher[sel] >= 0.5).astype('float32')
print(f'pseudo-labelled: {sel.sum():,} of {len(sel):,} Kaggle test images ({sel.mean():.1%}); '
      f'positives {y_pl.mean():.1%} (train positives {y_train.mean():.1%})')
X_st = np.concatenate([X_train, X_pl]); y_st = np.concatenate([y_train, y_pl])
del X_pl, X_train; gc.collect()   # free RAM (X_st holds the training images)
plt.figure(figsize=(6, 3)); plt.hist(teacher, bins=100); plt.yscale('log')
plt.axvline(PL_NEG, c='r'); plt.axvline(PL_POS, c='r'); plt.title('Teacher confidence on Kaggle test'); plt.show()
print(f'hours left: {hours_left():.2f}')

## 6. Pipeline with stronger stain / colour augmentation

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
RGB_FROM_HED = tf.constant([[0.65, 0.70, 0.29], [0.07, 0.99, 0.11], [0.27, 0.57, 0.78]], tf.float32)
HED_FROM_RGB = tf.linalg.inv(RGB_FROM_HED)
LOG_EPS = float(np.log(1e-6))

def hed_jitter(x):
    b = tf.shape(x)[0]
    s = tf.einsum('bhwc,cd->bhwd', tf.math.log(tf.maximum(x, 1e-6)) / LOG_EPS, HED_FROM_RGB)
    s = s * tf.random.uniform([b, 1, 1, 3], 1 - HED_ALPHA, 1 + HED_ALPHA) + tf.random.uniform([b, 1, 1, 3], -HED_BETA, HED_BETA)
    return tf.clip_by_value(tf.exp(tf.einsum('bhwd,dc->bhwc', s * LOG_EPS, RGB_FROM_HED)), 0.0, 1.0)

def augment_batch(x, y):
    x = tf.cast(x, tf.float32) / 255.0
    b = tf.shape(x)[0]
    x = tf.where(tf.random.uniform([b, 1, 1, 1]) < 0.5, tf.reverse(x, [2]), x)
    x = tf.where(tf.random.uniform([b, 1, 1, 1]) < 0.5, tf.reverse(x, [1]), x)
    x = tf.where(tf.random.uniform([b, 1, 1, 1]) < 0.5, tf.transpose(x, [0, 2, 1, 3]), x)
    x = hed_jitter(x)
    gray = tf.reduce_mean(x, axis=-1, keepdims=True)                       # saturation jitter
    x = gray + (x - gray) * tf.random.uniform([b, 1, 1, 1], *SAT_RANGE)
    x = x + tf.random.uniform([b, 1, 1, 1], -BRIGHT, BRIGHT)               # brightness
    mean = tf.reduce_mean(x, axis=[1, 2, 3], keepdims=True)                # contrast
    x = (x - mean) * tf.random.uniform([b, 1, 1, 1], *CONTRAST) + mean
    return tf.clip_by_value(x, 0.0, 1.0) * 255.0, y

def to_float(x, y):
    return tf.cast(x, tf.float32), y

def make_ds(X, y=None, training=False, batch=BATCH_SIZE):
    if y is None:
        y = np.zeros(len(X), dtype='float32')
    idx = tf.data.Dataset.range(len(X))
    if training:
        idx = idx.shuffle(len(X), seed=SEED, reshuffle_each_iteration=True)
    idx = idx.batch(batch)
    def gather(i):
        xb, yb = tf.numpy_function(lambda j: (X[j], y[j]), [i], [tf.uint8, tf.float32])
        xb.set_shape([None, IMG_SIZE, IMG_SIZE, 3]); yb.set_shape([None])
        return xb, yb
    return idx.map(gather, num_parallel_calls=AUTOTUNE).map(augment_batch if training else to_float,
                                                            num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

aug, _ = augment_batch(tf.constant(np.repeat(X_st[:1], 8, axis=0)), tf.zeros(8))
fig, axes = plt.subplots(1, 9, figsize=(18, 2.4)); axes[0].imshow(X_st[0]); axes[0].set_title('original')
for ax, im in zip(axes[1:], aug.numpy().astype('uint8')): ax.imshow(im)
for ax in axes: ax.axis('off')
plt.suptitle('v3 augmentation (stronger stain + colour jitter)'); plt.tight_layout(); plt.show()

cw = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_st.astype(int))
CLASS_WEIGHT = {0: float(cw[0]), 1: float(cw[1])}
train_ds, val_ds = make_ds(X_st, y_st, training=True), make_ds(X_val, y_val)

def get_metrics():
    return [keras.metrics.AUC(name='auc'), keras.metrics.BinaryAccuracy(name='accuracy')]

class TimeBudget(keras.callbacks.Callback):
    def __init__(self, deadline):
        super().__init__(); self.deadline = deadline; self.d = []
    def on_epoch_begin(self, e, logs=None): self.t = time.time()
    def on_epoch_end(self, e, logs=None):
        self.d.append(time.time() - self.t)
        if time.time() + max(self.d) > self.deadline:
            print(f'[TimeBudget] stop after epoch {e + 1}'); self.model.stop_training = True

def predict(model, X):
    base = make_ds(X, batch=256); preds = []
    for t in range(8):
        def tf_t(x, y, t=t):
            if t & 1: x = tf.reverse(x, [2])
            if t & 2: x = tf.reverse(x, [1])
            if t & 4: x = tf.transpose(x, [0, 2, 1, 3])
            return x
        preds.append(model.predict(base.map(tf_t), verbose=0).ravel())
    return np.mean(preds, axis=0)

def load_teacher(name):
    m = keras.models.load_model(find_file(f'{name}.keras'), compile=False)
    for l in m.layers:                       # full fine-tuning, BatchNorm frozen (as in v2)
        l.trainable = not isinstance(l, layers.BatchNormalization)
    return m

# preflight: load + 2 training steps + prediction for each student, so errors show up in minutes
for name in STUDENTS:
    m = load_teacher(name)
    m.compile(optimizer=keras.optimizers.Adam(PL_LR), loss='binary_crossentropy', metrics=get_metrics())
    m.fit(make_ds(X_st[:256], y_st[:256], training=True), epochs=1, verbose=0)
    p0 = predict(m, X_val[:64]); assert np.isfinite(p0).all()
    print(f'preflight {name}: OK'); del m; keras.backend.clear_session(); gc.collect()

## 7. Fine-tune the students on train + pseudo-labels (saved right after each model)

In [ ]:
P_VAL, P_TEST, P_KAG, results = {}, {}, {}, {}
for i, name in enumerate(STUDENTS):
    try:
        left = len(STUDENTS) - i
        budget_h = (hours_left() - RESERVE_AFTER_H * left) / left
        if budget_h < 0.4:
            print(f'skip {name}: not enough time'); continue
        print(f'\n===== student {name} | budget {budget_h:.2f} h =====')
        set_seed(SEED)
        model = load_teacher(name)
        p_before = predict(model, X_val)
        W = f'{OUT_DIR}/{name}_pl.weights.h5'
        model.compile(optimizer=keras.optimizers.Adam(PL_LR), loss='binary_crossentropy', metrics=get_metrics())
        cbs = [keras.callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=PL_PATIENCE, restore_best_weights=True, verbose=1),
               keras.callbacks.ReduceLROnPlateau(monitor='val_auc', mode='max', factor=0.5, patience=1, min_lr=MIN_LR, verbose=1),
               keras.callbacks.ModelCheckpoint(W, monitor='val_auc', mode='max', save_best_only=True, save_weights_only=True),
               TimeBudget(time.time() + budget_h * 3600)]
        h = model.fit(train_ds, validation_data=val_ds, epochs=PL_MAX_EPOCHS, class_weight=CLASS_WEIGHT, callbacks=cbs, verbose=2)
        if os.path.exists(W):
            model.load_weights(W)
        P_VAL[name], P_TEST[name], P_KAG[name] = predict(model, X_val), predict(model, X_test), predict(model, X_kag)
        tag = name.lower()
        results[name] = {'val AUC teacher': roc_auc_score(y_val, p_before), 'val AUC student': roc_auc_score(y_val, P_VAL[name]),
                         'test AUC student': roc_auc_score(y_test, P_TEST[name]), 'epochs': len(h.history['val_auc']),
                         'best_epoch': int(np.argmax(h.history['val_auc'])) + 1}
        print(name, {k: round(v, 5) if isinstance(v, float) else v for k, v in results[name].items()})
        pd.DataFrame({'id': kag_ids, 'label': P_KAG[name]}).to_csv(f'{OUT_DIR}/submission_pl_{tag}.csv', index=False)
        pd.DataFrame({'id': va_df['id'].values, 'label': y_val.astype(int), 'p': P_VAL[name]}).to_csv(f'{OUT_DIR}/val_pred_pl_{tag}.csv', index=False)
        pd.DataFrame({'id': te_df['id'].values, 'label': y_test.astype(int), 'p': P_TEST[name]}).to_csv(f'{OUT_DIR}/test_pred_pl_{tag}.csv', index=False)
        model.save(f'{OUT_DIR}/{name}_pl.keras')
        print(f'saved | hours left {hours_left():.2f}')
        del model; keras.backend.clear_session(); gc.collect()
    except Exception:
        print(f'!!! {name} failed - continuing'); traceback.print_exc(); keras.backend.clear_session(); gc.collect()

## 8. Final ensembles and summary

In [ ]:
names = list(P_KAG)
assert names, 'no student finished'
subs = {}
if len(names) > 1:
    subs['submission_pl_ensemble.csv'] = rank_avg([P_KAG[n] for n in names])
    P_VAL['PL ensemble'] = rank_avg([P_VAL[n] for n in names]); P_TEST['PL ensemble'] = rank_avg([P_TEST[n] for n in names])
# students + all four earlier models
subs['submission_pl_plus_all.csv'] = rank_avg([P_KAG[n] for n in names] + [KAG[m].values for m in MODELS4])
for f, p in subs.items():
    pd.DataFrame({'id': kag_ids, 'label': p}).to_csv(f'{OUT_DIR}/{f}', index=False)
summary = pd.DataFrame({n: {'val AUC': roc_auc_score(y_val, P_VAL[n]), 'test AUC': roc_auc_score(y_test, P_TEST[n])} for n in P_VAL}).T
display(summary.round(5)); summary.to_csv(f'{OUT_DIR}/summary_v3.csv')
with open(f'{OUT_DIR}/results_v3.json', 'w') as f:
    json.dump({'students': results, 'pseudo_labels': int(sel.sum()), 'clinical_threshold': thr_clin,
               'config': {'PL_POS': PL_POS, 'PL_NEG': PL_NEG, 'PL_LR': PL_LR, 'HED_ALPHA': HED_ALPHA, 'HED_BETA': HED_BETA}},
              f, indent=2, default=float)
print(f'Total run time {(time.time() - RUN_START) / 3600:.2f} h'); print(sorted(os.listdir(OUT_DIR)))